Dataset Preparation and Training



1. [Install Dependancies](#scrollTo=hl_32iDyTBcQ)
2. [CMU dataset preparation](#scrollTo=qW5g2oIwTHaC)
3. [Balabit dataset preparation](#scrollTo=uiw6T_X_TMmZ)
4. [Combine dataset](#scrollTo=PxgtiTGjTSd_)
5. [Training](#scrollTo=7cId0cP3TXAi)






#### Setup

In [ ]:
# Install dependencies
!pip install tensorflow numpy pandas scikit-learn scipy

In [ ]:
# Create folders
import os
os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("logs", exist_ok=True)
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/features", exist_ok=True)

#### CMU Dataset

In [ ]:
# Download CMU dataset directly
!wget -q "https://www.cs.cmu.edu/~keystroke/DSL-StrongPasswordData.csv" \
     -O data/cmu_raw.csv

# Verify it downloaded
import os
size = os.path.getsize("data/cmu_raw.csv")
print(f"Downloaded: {size/1024:.1f} KB")

Downloaded: 4560.5 KB


In [ ]:
# Convert CMU to model-ready format
import numpy as np
import pandas as pd

df = pd.read_csv("data/cmu_raw.csv")
print(f"Raw shape: {df.shape}")
print(f"Columns sample: {list(df.columns[:8])}")

timing_cols = [c for c in df.columns if c.startswith(('H.', 'DD.', 'UD.'))]
print(f"Timing columns found: {len(timing_cols)}")

sequences = []
for (subj, sess), group in df.groupby(['subject', 'sessionIndex']):
    vals = group[timing_cols].values.astype(np.float32)
    if len(vals) < 5:
        continue
    if len(vals) < 30:
        pad = np.zeros((30 - len(vals), vals.shape[1]), dtype=np.float32)
        vals = np.vstack([vals, pad])
    else:
        vals = vals[:30]
    if vals.shape[1] >= 18:
        vals = vals[:, :18]
    else:
        pad = np.zeros((30, 18 - vals.shape[1]), dtype=np.float32)
        vals = np.hstack([vals, pad])
    vals = np.clip(vals, -5, 5)
    sequences.append(vals)

sequences = np.stack(sequences).astype(np.float32)
print(f"Final dataset shape: {sequences.shape}")
np.save("data/cmu_sequences.npy", sequences)
print("Saved → data/cmu_sequences.npy")

Raw shape: (20400, 34)
Columns sample: ['subject', 'sessionIndex', 'rep', 'H.period', 'DD.period.t', 'UD.period.t', 'H.t', 'DD.t.i']
Timing columns found: 31
Final dataset shape: (408, 30, 18)
Saved → data/cmu_sequences.npy


#### Balabit dataset

In [ ]:
# Download Balabit dataset directly from GitHub
!git clone https://github.com/balabit/Mouse-Dynamics-Challenge.git data/balabit_raw
print("Done")

Cloning into 'data/balabit_raw'...
remote: Enumerating objects: 1711, done.
remote: Total 1711 (delta 0), reused 0 (delta 0), pack-reused 1711 (from 1)
Receiving objects: 100% (1711/1711), 42.60 MiB | 15.20 MiB/s, done.
Resolving deltas: 100% (3/3), done.
Updating files: 100% (1678/1678), done.
Done


In [ ]:
import numpy as np
import pandas as pd
import os

def extract_balabit_features(df, window_size=50):
    """
    Extract 18 behavioral features from Balabit mouse session data.
    Each window of 'window_size' rows becomes one time step.
    """
    # Keep only Move events for movement features
    moves = df[df['button'] == 'NoButton'].copy()
    clicks = df[df['state'].isin(['Pressed', 'Released'])].copy()

    features = []

    for i in range(0, len(moves) - window_size, window_size // 2):
        window = moves.iloc[i:i+window_size]
        click_window = clicks[
            (clicks['record timestamp'] >= window['record timestamp'].iloc[0]) &
            (clicks['record timestamp'] <= window['record timestamp'].iloc[-1])
        ]

        # Compute time deltas
        dt = window['record timestamp'].diff().fillna(0.1)

        # Compute dx, dy, speed
        dx = window['x'].diff().fillna(0)
        dy = window['y'].diff().fillna(0)
        dist = np.sqrt(dx**2 + dy**2)
        speed = dist / dt.replace(0, 0.001)

        # Acceleration
        acc = speed.diff().abs().fillna(0) / dt.replace(0, 0.001)

        # Path efficiency
        total_path = dist.sum()
        straight = np.sqrt((window['x'].iloc[-1] - window['x'].iloc[0])**2 +
                          (window['y'].iloc[-1] - window['y'].iloc[0])**2)
        path_eff = min(1.0, straight / total_path) if total_path > 1 else 1.0

        # Idle ratio (speed < 5 px/s)
        idle_ratio = (speed < 5).mean()

        # Click features
        n_clicks = len(click_window)
        window_duration = window['record timestamp'].iloc[-1] - window['record timestamp'].iloc[0]
        click_rate = n_clicks / max(window_duration, 0.1)

        # Double clicks (two clicks within 0.3s)
        if len(click_window) >= 2:
            click_ts = click_window['record timestamp'].values
            dc = sum(1 for a, b in zip(click_ts, click_ts[1:]) if b - a < 0.3)
        else:
            dc = 0
        double_click_rate = dc / max(window_duration, 0.1)

        # Scroll (Right button drag as proxy)
        scroll_events = df[df['button'] == 'Scroll'] if 'Scroll' in df['button'].values else pd.DataFrame()
        scroll_vel = len(scroll_events) / max(window_duration, 0.1)

        # Build 18-feature vector matching FEATURE_NAMES exactly
        feat = [
            speed.mean(),           # 0  typing_speed_wpm (proxy: mouse speed)
            dt.mean(),              # 1  keystroke_dwell_mean (proxy: time between events)
            dt.std(),               # 2  keystroke_dwell_std
            dist.mean(),            # 3  keystroke_flight_mean (proxy: move distance)
            dist.std(),             # 4  keystroke_flight_std
            0.0,                    # 5  error_rate (no keyboard data)
            0.0,                    # 6  backspace_ratio
            float(n_clicks),        # 7  burst_count (proxy: click count)
            float((speed < 1).sum()),# 8 pause_count_long (proxy: near-zero speed)
            speed.std() / (speed.mean() + 1e-9),  # 9  typing_rhythm_cv
            speed.mean(),           # 10 mouse_speed_mean
            speed.std(),            # 11 mouse_speed_std
            acc.mean(),             # 12 mouse_acceleration_mean
            click_rate,             # 13 click_rate
            double_click_rate,      # 14 double_click_rate
            scroll_vel,             # 15 scroll_velocity_mean
            idle_ratio,             # 16 mouse_idle_ratio
            path_eff,               # 17 mouse_path_efficiency
        ]
        features.append(feat)

    return np.array(features, dtype=np.float32)


def build_sequences(features, seq_len=30):
    """Slide over feature rows to build (N, 30, 18) sequences."""
    sequences = []
    for i in range(0, len(features) - seq_len + 1, seq_len // 2):
        seq = features[i:i+seq_len]
        if len(seq) == seq_len:
            # Clip extremes
            seq = np.clip(seq, -10, 10)
            sequences.append(seq)
    return sequences


# ── Process all training users ────────────────────────────────────────────────
training_dir = "data/balabit_raw/training_files"
all_sequences = []
skipped = 0

for user in sorted(os.listdir(training_dir)):
    user_path = os.path.join(training_dir, user)
    if not os.path.isdir(user_path):
        continue

    user_seqs = []
    for session_file in os.listdir(user_path):
        fpath = os.path.join(user_path, session_file)
        try:
            df = pd.read_csv(fpath)
            if len(df) < 200:
                skipped += 1
                continue
            feats = extract_balabit_features(df)
            if len(feats) >= 30:
                seqs = build_sequences(feats, seq_len=30)
                user_seqs.extend(seqs)
        except Exception as e:
            print(f"  Skipped {session_file}: {e}")
            skipped += 1

    print(f"  {user}: {len(user_seqs)} sequences")
    all_sequences.extend(user_seqs)

print(f"\nTotal sequences: {len(all_sequences)}")
print(f"Skipped files: {skipped}")

# ── Save ──────────────────────────────────────────────────────────────────────
balabit_arr = np.stack(all_sequences).astype(np.float32)
print(f"Final shape: {balabit_arr.shape}")  # should be (N, 30, 18)

np.save("data/balabit_sequences.npy", balabit_arr)
print("Saved → data/balabit_sequences.npy")

  user12: 581 sequences
  user15: 341 sequences
  user16: 528 sequences
  user20: 646 sequences
  user21: 280 sequences
  user23: 283 sequences
  user29: 285 sequences
  user35: 228 sequences
  user7: 1042 sequences
  user9: 1036 sequences

Total sequences: 5250
Skipped files: 0
Final shape: (5250, 30, 18)
Saved → data/balabit_sequences.npy


#### Combine Dataset

In [ ]:
import numpy as np

cmu     = np.load("data/cmu_sequences.npy")
balabit = np.load("data/balabit_sequences.npy")

print(f"CMU:     {cmu.shape}")
print(f"Balabit: {balabit.shape}")

combined = np.concatenate([cmu, balabit], axis=0)

# Shuffle
idx = np.random.default_rng(42).permutation(len(combined))
combined = combined[idx]

np.save("data/cmu_sequences.npy", combined)  # overwrite so model.py picks it up
print(f"Combined and saved: {combined.shape}")

CMU:     (408, 30, 18)
Balabit: (5250, 30, 18)
Combined and saved: (5658, 30, 18)


#### Training

In [ ]:
import tensorflow as tf
import json

# Load the already-trained model
# Add custom_objects to handle potential deserialization issues for 'mse'
custom_objects = {"mse": tf.keras.metrics.MeanSquaredError}
model = tf.keras.models.load_model('/content/models/autoencoder.h5', custom_objects=custom_objects)

# Convert with SELECT_TF_OPS (fixes the TensorListReserve error)
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,
]
converter._experimental_lower_tensor_list_ops = False

tflite_model = converter.convert()

with open('/content/models/autoencoder.tflite', 'wb') as f:
    f.write(tflite_model)

print(f"Done — {len(tflite_model)/1024:.1f} KB")

In [ ]:
# Upload model.py and config.py first via the Files panel,then run training
!python model.py data models

2026-05-26 11:35:18.994631: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)
2026-05-26 11:35:19,171 [__main__] INFO Autoencoder built: 117026 params
Model: "lstm_autoencoder"
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 30, 18)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_enc_1 (Bidirectional)        │ (None, 30, 128)        │        42,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_drop_1 (Dropout)            │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_enc_2 (Bidirectional

In [ ]:
# Verify all three files exist before downloading
import os
for f in ['models/autoencoder.h5', 'models/autoencoder.tflite', 'models/threshold.json']:
    exists = os.path.exists(f)
    size   = os.path.getsize(f)/1024 if exists else 0
    print(f"{'✓' if exists else '✗'}  {f}  ({size:.1f} KB)")

✓  models/autoencoder.h5  (1457.6 KB)
✓  models/autoencoder.tflite  (222.6 KB)
✓  models/threshold.json  (0.1 KB)


In [ ]:
from model import build_autoencoder
m = build_autoencoder()
m.summary()

Model: "lstm_autoencoder"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input (InputLayer)              │ (None, 30, 18)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_enc_1 (Bidirectional)        │ (None, 30, 128)        │        42,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_drop_1 (Dropout)            │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bi_enc_2 (Bidirectional)        │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ enc_drop_2 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ latent (Dense)                  │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ repeat (RepeatVector)           │ (None, 30, 16)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_lstm_1 (LSTM)               │ (None, 30, 32)         │         6,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_drop_1 (Dropout)            │ (None, 30, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_lstm_2 (LSTM)               │ (None, 30, 64)         │        24,832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dec_drop_2 (Dropout)            │ (None, 30, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ reconstruction                  │ (None, 30, 18)         │         1,170 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 117,026 (457.13 KB)

 Trainable params: 117,026 (457.13 KB)

 Non-trainable params: 0 (0.00 B)